In [65]:


import mrc
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
import sys
sys.path.append(r"C:\Users\sjefs\Desktop\Uni\Speciale\Kode\library")
import CV_functions as CV_func
import CV_classes as CV_class

# 3D ideas

It seems like there is a lot of foci in different z-planes, so we need to move to 3D. Here we have three approaches by how difficult they are. 
1. Project the z-stack into a 2D image (e.g., using maximum intensity projection) and analyze foci in 2D.
2. Combine information from all z-planes to analyze foci in a more comprehensive 3D context.
3. Treat each z-stack as a separate 3D image and analyze foci in 3D.

The classes would probably need a lot of optimization if they should run on all this data

# Data format

So far it seems like the images are seperated so every other image belongs to channel 2. Then there should be 20 images in the z-stack for each channel.

That means one timepoint has 40 images. 20 in each channel. 

The .log file contains information regarding timestamps and total number of frames for each timepoint.

# Loading and formating raw data

- Loading and slicing/reshaping takes around 0.5 seconds from the .dv file
- Each channel is saved in a numpy .npy file in around 10 seconds per channel
- Loading the saved .npy files takes only a fraction of a second

so a speed up can be archieved, by sacrificing some saving time and using the .npy files instead of the original .dv file

I structure it so a channel can be acessed as channel_1[time, z, x, y]

In [6]:
def process_dv_file(run_number, savenp = False):
    raw_folder = r"C:\Users\sjefs\Desktop\Uni\Speciale\Data\SharangGarudTopBP1data_raw\\"
    processed_folder = r"C:\Users\sjefs\Desktop\Uni\Speciale\Data\SharangGarudTopBP1data_processed\\"

    filename = f"topbp1-timelapse-jf646-{run_number}_R3D.dv"
    path_data = raw_folder + filename
    path_brightfield = raw_folder + filename[:-3] + "_REF.dv"

    image = mrc.load(path_data)

    channel_1 = image[::2].reshape(-1, 20, *image.shape[1:])
    channel_2 = image[1::2].reshape(-1, 20, *image.shape[1:])

    if savenp:
        np.save(processed_folder + f"TopBP1_run_{run_number}_channel_1.npy", channel_1)
        np.save(processed_folder + f"TopBP1_run_{run_number}_channel_2.npy", channel_2)

    brightfield = mrc.load(path_brightfield)
    if savenp:
        np.save(processed_folder + f"TopBP1_run_{run_number}_brightfield.npy", brightfield)

    return channel_1, channel_2


#run_number = "1_1"
#channel_1, channel_2 = process_dv_file(run_number)


## processing all files

In [5]:
import os

processed_folder = r"C:\Users\sjefs\Desktop\Uni\Speciale\Data\SharangGarudTopBP1data_processed\\"

numbers = (
    [f"{i}_{j}" for i in range(1, 14) for j in (1, 2)]
    + ["1"]
)

for run_number in numbers:
    # check if the processed file already exists
    if os.path.exists(processed_folder + f"TopBP1_run_{run_number}_brightfield.npy"):
        print(f"Processed file for run_number {run_number} already exists")
        continue

    try:
        process_dv_file(run_number)
    except Exception as e:
        print(f"Failed to process run_number {run_number}: {e}")
    else:
        print(f"Successfully processed run_number {run_number}")

Processed file for run_number 1_1 already exists
Processed file for run_number 1_2 already exists
Processed file for run_number 2_1 already exists
Processed file for run_number 2_2 already exists
Processed file for run_number 3_1 already exists
Processed file for run_number 3_2 already exists
Processed file for run_number 4_1 already exists
Processed file for run_number 4_2 already exists
Processed file for run_number 5_1 already exists


KeyboardInterrupt: 

In [70]:
import importlib
importlib.reload(CV_class)

<module 'CV_classes' from 'C:\\Users\\sjefs\\Desktop\\Uni\\Speciale\\Kode\\library\\CV_classes.py'>

# Convert to maximum intensity projection

In [79]:
def save_max_projections(run_number):
    channel_1, channel_2 = process_dv_file(run_number)
    path = r"C:\Users\sjefs\Desktop\Uni\Speciale\Data\SharangGarudTopBP1data_processed\TopBP1_zproj"

    for channel_name, channel in (
        ("channel_1", channel_1),
        ("channel_2", channel_2),
    ):

        folder = os.path.join(
            path, f"{channel_name}_run_{run_number}_zproj"
        )
        os.makedirs(folder, exist_ok=True)
# Save each frame of the maximum projection as a separate .npy file.
        # Channel shape: (time, z, x, y); project across z.
        projections = channel.max(axis=1)

        for frame_index, projection in enumerate(projections):
            filename = (
                f"{channel_name}_run_{run_number}"
                f"_zproj_t_{frame_index}.npy"
            )
            np.save(os.path.join(folder, filename), projection)

numbers = (
    [f"{i}_{j}" for i in range(1, 14) for j in (1, 2)]
    + ["1"]
)

for number in numbers:
    save_max_projections(number)


# Loading data zproj

In [ ]:
path_to_folder = r"C:\Users\sjefs\Desktop\Uni\Speciale\Data\SharangGarudTopBP1data_processed\TopBP1_zproj"
run_number = "1_1"
channel = 2

def load_time_series(channel, run_number, path_to_folder: str):
    subfolder_name = f"channel_{channel}_run_{run_number}_zproj"
    full_path = os.path.join(path_to_folder, subfolder_name)
    image_array = []

    for filename in sorted(os.listdir(full_path)):
        if filename.endswith(".npy"):
                image_array.append(CV_class.Picture(os.path.join(full_path, filename)))


    # Sort image according to their timestamp
    image_array.sort(key=lambda x: x.t)
    return image_array
# Test it

test = load_time_series(channel, run_number, path_to_folder)

CV_func.interactive_plot(test)

interactive(children=(IntSlider(value=0, description='i', max=25), Output()), _dom_classes=('widget-interact',…

# Loading metadata

In [86]:
from pathlib import Path

log_path = Path(
    r"C:\Users\sjefs\Desktop\Uni\Speciale\Data\SharangGarudTopBP1data_raw\topbp1-timelapse-jf646-1_1_R3D.dv.log"
)

def extract_metadata(key):
    saved_lines = []
    with log_path.open(encoding="utf-8") as file:
        for line in file:
            if key in line:
                saved_lines.append(line.strip())
    return [line.split(":")[-1].strip() for line in saved_lines]

keys = ["Exposure time", "Time Point" ]
exposure_times = extract_metadata("Exposure time")
time_points = extract_metadata("Time Point")
print(exposure_times)
print(time_points)


['0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 secs', '0.300000 secs', '0.200000 sec